<a href="https://colab.research.google.com/github/mariya7040ai/-arabic-english-medqa-uncertainty/blob/dev/analyze_all.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import json, math, os
import numpy as np, pandas as pd
from scipy.optimize import minimize_scalar
from sklearn.metrics import roc_auc_score

BASE = "/content/drive/MyDrive/medqa_results/results/"
RUNS = {
    ("Qwen2.5-7B", "en"): "Qwen2.5-7B-Instruct_en_no_prefill_seed42_n2816.jsonl",
    ("Qwen2.5-7B", "ar"): "Qwen2.5-7B-Instruct_ar_no_prefill_seed42_n4927.jsonl",
    ("Jais-adapted-7B", "en"): "jais-adapted-7b-chat_en_no_prefill_union_seed42_n2816.jsonl",
    ("Jais-adapted-7B", "ar"): "jais-adapted-7b-chat_ar_no_prefill_union_seed42_n4927.jsonl",
}
ALPHAS = [0.05, 0.10, 0.20]
R = 100          # conformal splits
B = 500          # bootstrap draws
SEED = 42

for k, f in RUNS.items():
    print(k, "exists:", os.path.exists(BASE + f))

Mounted at /content/drive
('Qwen2.5-7B', 'en') exists: True
('Qwen2.5-7B', 'ar') exists: True
('Jais-adapted-7B', 'en') exists: True
('Jais-adapted-7B', 'ar') exists: True


In [2]:
def ece_fast(conf, correct, n_bins=15):
    idx = np.digitize(conf, np.linspace(0, 1, n_bins + 1)[1:-1])
    tot = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            tot += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(tot)

def analyze(path):
    res = pd.DataFrame([json.loads(l) for l in open(path, encoding="utf-8")])
    S = [np.asarray(s, dtype=float) for s in res["scores"]]
    Y = res["answer"].values.astype(int); n = len(Y)
    K_arr = res["K"].values
    Kmax = K_arr.max()
    S_pad = np.full((n, Kmax), -np.inf)
    for i, s in enumerate(S):
        S_pad[i, :len(s)] = s
    valid = np.isfinite(S_pad)

    def probs_T(T, rows=None):
        Sp = S_pad if rows is None else S_pad[rows]
        Z = Sp / T
        Z = Z - np.max(Z, axis=1, keepdims=True)
        E = np.exp(Z)
        return E / E.sum(axis=1, keepdims=True)

    def fit_T(idx):
        def nll(logT):
            P = probs_T(np.exp(logT), rows=idx)
            return -np.mean(np.log(P[np.arange(len(idx)), Y[idx]] + 1e-12))
        return float(np.exp(minimize_scalar(nll, bounds=(-3, 5), method="bounded").x))

    def conformal_q(cal_scores, alpha):
        m = len(cal_scores)
        level = min(1.0, math.ceil((m + 1) * (1 - alpha)) / m)
        return float(np.quantile(cal_scores, level, method="higher"))

    rng = np.random.default_rng(SEED)
    out = {"n": n, "chance": float((1 / K_arr).mean()), "mass_on_letters": float(res["mass"].mean())}

    # raw metrics
    P1 = probs_T(1.0)
    conf = P1.max(axis=1); pred = np.where(valid, S_pad, -np.inf).argmax(axis=1); corr = (pred == Y).astype(float)
    onehot = np.zeros_like(P1); onehot[np.arange(n), Y] = 1.0
    brier_i = ((P1 - onehot) ** 2).sum(axis=1)
    out["accuracy"] = corr.mean(); out["mean_conf"] = conf.mean()
    out["ece_raw"] = ece_fast(conf, corr); out["brier"] = brier_i.mean()
    out["brier_uniform"] = float((1 - 1 / K_arr).mean())
    accs, eces = [], []
    for _ in range(B):
        ix = rng.integers(0, n, n); accs.append(corr[ix].mean()); eces.append(ece_fast(conf[ix], corr[ix]))
    out["acc_lo"], out["acc_hi"] = np.percentile(accs, [2.5, 97.5])
    out["ece_lo"], out["ece_hi"] = np.percentile(eces, [2.5, 97.5])

    # AUROC of confidence for correctness
    Sv = np.where(valid, S_pad, -np.inf)
    srt = np.sort(Sv, axis=1)[:, ::-1]; margin = srt[:, 0] - srt[:, 1]
    out["auroc_margin"] = roc_auc_score(corr, margin)
    au = []
    for _ in range(B):
        ix = rng.integers(0, n, n); au.append(roc_auc_score(corr[ix], margin[ix]))
    out["auroc_lo"], out["auroc_hi"] = np.percentile(au, [2.5, 97.5])

    # selective prediction (margin ranking)
    order = np.argsort(-margin, kind="stable"); err = 1.0 - corr[order]
    kk = np.arange(1, n + 1); risk = np.cumsum(err) / kk
    out["aurc"] = risk.mean()
    for c in [0.2, 0.4, 0.6, 0.8]:
        out[f"risk@{c}"] = float(risk[int(round(c * n)) - 1])
    out["error_overall"] = 1 - corr.mean()

    # temperature scaling over 20 random 50/50 splits
    Ts, er, et = [], [], []
    for _ in range(20):
        perm = rng.permutation(n); half = n // 2; cal, te = perm[:half], perm[half:]
        T = fit_T(cal); Pt = probs_T(T, rows=te)
        Ts.append(T); er.append(ece_fast(conf[te], corr[te])); et.append(ece_fast(Pt.max(axis=1), corr[te]))
    out["T_mean"], out["T_sd"] = np.mean(Ts), np.std(Ts, ddof=1)
    out["ece_raw_20"], out["ece_ts_20"], out["ece_ts_sd"] = np.mean(er), np.mean(et), np.std(et, ddof=1)

    # split conformal prediction over repeated 25/25/50 splits
    recs = []
    for r in range(R):
        perm = rng.permutation(n); n_fit, n_cal = int(0.25 * n), int(0.25 * n)
        fit, cal, test = perm[:n_fit], perm[n_fit:n_fit + n_cal], perm[n_fit + n_cal:]
        T = fit_T(fit)
        for variant, Tv in [("raw", 1.0), ("scaled", T)]:
            P = probs_T(Tv); s_all = 1.0 - P[np.arange(n), Y]
            for a in ALPHAS:
                q = conformal_q(s_all[cal], a)
                inset = valid[test] & ((1.0 - P[test]) <= q + 1e-12)
                hit = inset[np.arange(len(test)), Y[test]]; size = inset.sum(axis=1)
                recs.append(dict(variant=variant, alpha=a, coverage=hit.mean(), size=size.mean(),
                                 size_over_K=(size / K_arr[test]).mean(), singleton=(size == 1).mean(),
                                 all_options=(size == K_arr[test]).mean()))
    conf_df = pd.DataFrame(recs).groupby(["variant", "alpha"]).mean().reset_index()
    return out, conf_df

summ_rows, conf_rows = [], []
for (model, lang), fname in RUNS.items():
    print("analyzing", model, lang, "...")
    out, cdf = analyze(BASE + fname)
    out.update(model=model, language=lang); summ_rows.append(out)
    cdf.insert(0, "language", lang); cdf.insert(0, "model", model); conf_rows.append(cdf)

summary = pd.DataFrame(summ_rows).set_index(["model", "language"])
conformal = pd.concat(conf_rows).set_index(["model", "language", "variant", "alpha"])
summary.round(4).to_csv("/content/drive/MyDrive/medqa_results/summary_metrics.csv")
conformal.round(4).to_csv("/content/drive/MyDrive/medqa_results/summary_conformal.csv")
print("saved summary_metrics.csv and summary_conformal.csv to Drive")

analyzing Qwen2.5-7B en ...
analyzing Qwen2.5-7B ar ...
analyzing Jais-adapted-7B en ...
analyzing Jais-adapted-7B ar ...
saved summary_metrics.csv and summary_conformal.csv to Drive


In [3]:
# Table 1: calibration, discrimination, selective prediction
cols = ["n", "chance", "accuracy", "acc_lo", "acc_hi", "mass_on_letters", "mean_conf", "ece_raw", "ece_lo", "ece_hi",
        "ece_raw_20", "ece_ts_20", "T_mean", "T_sd", "brier", "brier_uniform", "auroc_margin", "auroc_lo", "auroc_hi",
        "error_overall", "risk@0.2", "risk@0.4", "risk@0.6", "risk@0.8", "aurc"]
print(summary[cols].round(3).T.to_string())

model           Qwen2.5-7B           Jais-adapted-7B          
language                en        ar              en        ar
n                 2816.000  4927.000        2816.000  4927.000
chance               0.250     0.240           0.250     0.240
accuracy             0.561     0.430           0.369     0.308
acc_lo               0.541     0.417           0.352     0.296
acc_hi               0.582     0.444           0.386     0.321
mass_on_letters      1.000     1.000           0.899     0.926
mean_conf            0.948     0.916           0.611     0.618
ece_raw              0.386     0.486           0.242     0.309
ece_lo               0.367     0.472           0.226     0.297
ece_hi               0.407     0.500           0.260     0.322
ece_raw_20           0.388     0.487           0.244     0.308
ece_ts_20            0.068     0.035           0.042     0.042
T_mean               8.710    10.904           3.398     5.699
T_sd                 0.231     0.342           0.174   

In [5]:
# Table 2: conformal prediction (alpha = target error; set size shown in options and as a share of the options)
print(conformal.round(3).to_string())

                                        coverage   size  size_over_K  singleton  all_options
model           language variant alpha                                                      
Qwen2.5-7B      en       raw     0.05      0.954  2.993        0.748      0.203        0.533
                                 0.10      0.905  2.514        0.629      0.303        0.331
                                 0.20      0.802  1.881        0.470      0.482        0.112
                         scaled  0.05      0.953  2.959        0.740      0.152        0.445
                                 0.10      0.905  2.510        0.627      0.234        0.246
                                 0.20      0.802  1.851        0.463      0.414        0.045
                ar       raw     0.05      0.952  3.650        0.868      0.067        0.700
                                 0.10      0.902  3.251        0.773      0.130        0.517
                                 0.20      0.802  2.616        0.622  